# Register Private Application Gateway Feature

This notebook demonstrates how to register the feature [required to deploy a Private Application Gateway](learn.microsoft.com/en-us/azure/application-gateway/application-gateway-private-deployment?tabs=portal). The subnet you plan to deploy the application gateway instances to must be [delegated to Microsoft.Network/applicationGateways](Microsoft.Network/applicationGateways). Manual registration of this feature is still required as of 4/24/2026.

The user running this notebook must hold the Azure RBAC built-in Owner permission to register the feature.

## Setup the notebook

Before running the notebook you must create a .env file and populate the required variables. Reference the env-sample file for a reference.

The cell in this notebook acquired an access token from Entra ID for the user which is used to run the commands in the notebook.

### Obtain an access token for the user

You must log into az cli prior to running this cell.

In [ ]:
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(".env", override=True)

# Get a token for Microsoft Graph with the logged in user
credential = DefaultAzureCredential()
scopes = ["https://management.azure.com/.default"]

user_token = credential.get_token(*scopes)

## Register the feature

The cells in this section registers the feature in the subscription.

In [ ]:
import os
import time
import requests
from dotenv import load_dotenv

# Load environment variables from .env file and set variables
load_dotenv(".env", override=True)
AZURE_SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")

def register_private_app_gw_feature(subscription: str, token: str):
    """This function registers the private application gateway feature for the subscription.
    Args:
        subscription (str): The subscription ID to register the feature for.
        token (str): The access token to authenticate the request.
    Returns:
        bool: True if the feature is registered or in the process of registering, False otherwise.
    Raises:
        requests.exceptions.HTTPError: If the API request fails.
    """
    url = f"https://management.azure.com/subscriptions/{subscription}/providers/Microsoft.Features/providers/Microsoft.Network/features/EnableApplicationGatewayNetworkIsolation/register"
    api_version = "2021-07-01"

    response = requests.post(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    response.raise_for_status()

    # Check the registration status of the feature
    state = response.json().get("properties", {}).get("state", "unknown")
    print(f"Registration state: {state}")
    return state in ("Registered", "Registering")

def get_feature_registration_status(subscription: str, feature: str, token: str):
    """This function checks the registration status of the private application gateway feature for the subscription.
    Args:
        subscription (str): The subscription ID to check the feature registration status for.
        feature (str): The feature name to check the registration status for.
        token (str): The access token to authenticate the request.
    Returns:
        str: The registration state of the feature.
    Raises:
        requests.exceptions.HTTPError: If the API request fails.
    """
    url = f"https://management.azure.com/subscriptions/{subscription}/providers/Microsoft.Features/providers/Microsoft.Network/features/{feature}"
    api_version = "2021-07-01"

    response = requests.get(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    # Check if the request was successful and raise an error if not
    response.raise_for_status()

    # Extract and return the registration state from the response
    state = response.json().get("properties", {}).get("state", "Unknown")
    print(f"The feature {feature} registration state: {state}")
    return state

register_private_app_gw_feature(
    subscription=AZURE_SUBSCRIPTION_ID,
    token=user_token.token
)

while True:
    status = get_feature_registration_status(
        subscription=AZURE_SUBSCRIPTION_ID,
        feature="EnableApplicationGatewayNetworkIsolation",
        token=user_token.token
    )
    if status == "Registered":
        print(f"The feature EnableApplicationGatewayNetworkIsolation is registered.")
        break
    elif status == "Registering":
        print(f"The feature EnableApplicationGatewayNetworkIsolation is still registering. Checking again in 10 seconds...")
        time.sleep(10)
    else:
        print(f"Unexpected registration state: {status}")
        break
